%conda install -n langagent ipykernel --update-deps --force-reinstall

%pip install certifi pydantic httpx sniffio distro requests jsonpatch langchain-core multidict attrs yarl aiohappyeyeballs aiosignal langchain-ollama langchain-tavily

In [15]:
import os
import certifi
from dotenv import load_dotenv

from langchain_ollama import ChatOllama
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub

from langchain.agents import create_react_agent, AgentExecutor

In [16]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [17]:
from langchain_tavily import TavilySearch
search_tool = TavilySearch(
    api_key=TAVILY_API_KEY,
    max_results=5
)
result = search_tool.invoke("Give me the latest news on AI?")
result

{'query': 'Give me the latest news on AI?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.artificialintelligence-news.com',
   'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
   'content': 'October 30, 2025\n\n### Malaysia launches Ryt Bank, its first AI-powered bank\n\nFinance AI\n\nAugust 26, 2025\n\n### Google’s Veo 3 AI video creation tools are now widely available\n\nAI in Action\n\nJuly 29, 2025\n\n#### Computer Vision\n\n### Microsoft’s Majorana 2 quantum chip is also a case study for agentic AI in R&D\n\nInside AI\n\nJune 3, 2026\n\n### US and Japan announce sweeping AI and tech collaboration\n\nArtificial Intelligence\n\nApril 11, 2024\n\n### UK and Canada sign AI compute agreement [...] August 18, 2026\n\n### Okta targets AI agent token costs with MCP scoping\n\nData Engineering & MLOps\n\nAugust 13, 2026\n\n### Red Hat, NVIDIA, IBM back project turning AI policy into code\n\nGovernance, Regulat

In [18]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.7
)
response = llm.invoke("What year is it?")
response

AIMessage(content="I do not have access to real-time date and time information. Therefore, I cannot tell you the exact current year.\n\nPlease check a reliable source, such as a clock, calendar, or search engine, for today's date.", additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-09-08T01:19:20.2567298Z', 'done': True, 'done_reason': 'stop', 'total_duration': 84073587800, 'load_duration': 14888800, 'prompt_eval_count': 21, 'prompt_eval_duration': 7650376000, 'eval_count': 244, 'eval_duration': 75089656000, 'logprobs': None, 'model_name': 'gemma4:e4b'}, id='run--01a07e97-da3e-7f82-8969-c1f75cc66fdd-0', usage_metadata={'input_tokens': 21, 'output_tokens': 244, 'total_tokens': 265})

In [19]:
from langsmith import Client
client = Client()
prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True,
)
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [20]:
tools = [search_tool]

In [24]:
agent = create_react_agent(
    llm=llm,
    tools=[search_tool],
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True,
    handle_parsing_errors=True
)

In [ ]:
response = agent_executor.invoke({
    "input": (
        "What's the latest news on Dangote Refinery"
        "How can an entrepreneur in Tech and AI benefit from this news?"
    )
})
response["output"]



> Entering new AgentExecutor chain...
Thought: The user is asking for the latest news on the Dangote Refinery and how an entrepreneur in Tech and AI can benefit from it. I need to use the `tavily_search` tool to gather the most current information on the refinery first. I will formulate a search query that targets recent developments.
Action: tavily_search
Action Input: latest news on Dangote Refinery{'query': 'latest news on Dangote Refinery', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://finance.yahoo.com/markets/stocks/articles/nigerias-dangote-refinery-looks-raise-173302382.html', 'title': "Nigeria's Dangote refinery looks to raise $1.6 bn in IPO", 'content': 'Lagos-based Dangote Petroleum Refinery is aiming to raise funds to double its capacity to 1.4 million barrels a day, which would make it the world\'s largest refinery, surpassing Jamnagar in India.\n\nWhen it goes public, the group will offer "4.1 billion ordinary shares at 525 nair